# Baselines và Augmentation (IBM Focus)

## 1) Thiết lập môi trường và đường dẫn

- Thiết lập môi trường tái lập kết quả (seed cố định) và định nghĩa đường dẫn dữ liệu/kết quả.
- Việc cố định seed là bắt buộc để các lần chạy 5-fold cho ra kết quả ổn định, dễ đối chiếu.
- RESULTS_DIR được tạo sẵn để tránh lỗi khi lưu metrics ở cuối notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/metrics")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: F:\ML - DL Projects\Deep Learning Employee Attrition BiTCN\data\processed
Results dir: F:\ML - DL Projects\Deep Learning Employee Attrition BiTCN\results\metrics


## 2) Kiểm tra dependency thí nghiệm

- Kiểm tra khả dụng của các thư viện quan trọng (imblearn, xgboost, torch).
- Cách làm này giúp notebook linh hoạt: nếu thiếu thư viện, pipeline vẫn chạy ở mức tối thiểu thay vì dừng toàn bộ.
- Khi chạy thực nghiệm đầy đủ (SMOTE, ADASYN, GAN, DL), trạng thái các cờ HAS_* cần là True.

In [2]:
# Dependency checks (skip model/augmentation gracefully if package missing)
HAS_IMBLEARN = True
HAS_XGBOOST = True
HAS_TORCH = True

try:
    from imblearn.over_sampling import SMOTE, ADASYN
except Exception as e:
    HAS_IMBLEARN = False
    print("imblearn not available:", e)

try:
    from xgboost import XGBClassifier
except Exception as e:
    HAS_XGBOOST = False
    print("xgboost not available:", e)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    torch.manual_seed(SEED)
except Exception as e:
    HAS_TORCH = False
    print("torch not available:", e)

print("HAS_IMBLEARN:", HAS_IMBLEARN)
print("HAS_XGBOOST:", HAS_XGBOOST)
print("HAS_TORCH:", HAS_TORCH)

HAS_IMBLEARN: True
HAS_XGBOOST: True
HAS_TORCH: True


## 3) Nạp dữ liệu processed và kiểm tra lệch lớp

- Nạp 2 bộ dữ liệu đã processed từ 1_EDA_Preprocessing và kiểm tra phân phối nhãn ban đầu.
- Kết quả phân phối nhãn là cơ sở để quyết định cần augmentation (đặc biệt với IBM có lệch lớp rõ).
- Nếu file đầu vào thay đổi schema, đây là nơi phát hiện sớm nhất trước khi vào vòng lặp CV.

In [3]:
# Load processed data exported from Notebook 1
kaggle_df = pd.read_csv(DATA_DIR / "Kaggle_Cleaned.csv")
ibm_df = pd.read_csv(DATA_DIR / "IBM_Cleaned.csv")

print("Kaggle shape:", kaggle_df.shape)
print("IBM shape:", ibm_df.shape)

print("\nKaggle target distribution (Churn):")
print(kaggle_df["Churn"].value_counts())

print("\nIBM target distribution (Attrition):")
print(ibm_df["Attrition"].value_counts())

Kaggle shape: (10000, 32)
IBM shape: (1470, 44)

Kaggle target distribution (Churn):
Churn
0    7972
1    2028
Name: count, dtype: int64

IBM target distribution (Attrition):
Attrition
0    1233
1     237
Name: count, dtype: int64


## 4) Các hàm Utility và thiết lập 5-Fold Cross-Validation

- Core cell cho thiết kế thí nghiệm: định nghĩa số fold, thông số augmentation và các hàm metrics.
- Hàm get_xy chủ động loại cột ID-like và ép feature về numeric để tránh lỗi định dạng khi train.
- Hàm build_cml_models tạo bộ baseline CML nhất quán cho mọi fold, giúp so sánh công bằng giữa các phương pháp augmentation.

In [4]:
# Core utilities
N_SPLITS = 5
AUG_METHODS = ["raw", "smote", "adasyn", "gan"]
GAN_TARGET_TOTAL = 10000
GAN_TARGET_PER_CLASS = GAN_TARGET_TOTAL // 2
GAN_LATENT_DIM = 16
GAN_EPOCHS = 120
GAN_BATCH_SIZE = 64
DL_EPOCHS = 25
DL_BATCH_SIZE = 64


def get_xy(df: pd.DataFrame, target_col: str):
    id_like = [c for c in df.columns if c.lower() in {"employee id", "employeeid", "employee_number"} or c.lower().endswith("id")]
    X = df.drop(columns=[target_col] + id_like, errors="ignore").copy()
    y = df[target_col].astype(int).copy()

    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        raise ValueError(f"Non-numeric features found: {non_numeric}")

    return X, y


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
    }


def get_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    pred = model.predict(X)
    return np.asarray(pred).astype(float)


def build_cml_models():
    models = {
        "LogisticRegression": LogisticRegression(max_iter=3000, random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    }
    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            random_state=SEED,
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1,
        )
    return models

## 5) Hàm Cross-Validation tổng quát cho một mô hình

- Hàm này đúng yêu cầu của bài toán: nhận trực tiếp một mô hình và trả về trung bình metrics qua 5-fold.
- Điểm quan trọng chống data leakage nằm ở chỗ augmentation chỉ áp dụng cho X_train, y_train trong từng fold.
- Có thể dùng hàm này để benchmark nhanh từng mô hình riêng lẻ trước khi chạy full pipeline.

In [5]:
def cv_score_single_model(model, X, y, augmentation="raw", n_splits=N_SPLITS, random_state=SEED):
    """Leakage-safe CV: augmentation is applied only on each training fold."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_metrics = []

    for tr_idx, te_idx in skf.split(X, y):
        X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
        X_test, y_test = X.iloc[te_idx], y.iloc[te_idx]

        X_aug, y_aug = augment_train_data(X_train, y_train, method=augmentation)

        clf = clone(model)
        clf.fit(X_aug, y_aug)
        y_prob = get_proba(clf, X_test)
        fold_metrics.append(compute_metrics(y_test, y_prob))

    return pd.DataFrame(fold_metrics).mean().to_dict()

## 6) Data Augmentation: SMOTE, ADASYN, GAN (IBM Focus)

- Hiện thực GAN theo đúng tinh thần bài báo (latent 16, Dense + LeakyReLU, đầu ra Generator là Tanh, Discriminator là Sigmoid).
- Augmentation GAN được thực hiện trong từng fold train và cân bằng về mục tiêu gần 1500/1500 cho IBM.
- Cùng cell này cũng gom logic SMOTE, ADASYN vào augment_train_data, giúp so sánh phương pháp augmentation theo cùng một giao diện.

In [6]:
# GAN utilities (IBM focus, PyTorch)
class Generator(nn.Module):
    def __init__(self, input_dim, latent_dim=GAN_LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, input_dim),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


def train_gan_and_generate(real_data, n_generate, latent_dim=GAN_LATENT_DIM, epochs=GAN_EPOCHS, batch_size=GAN_BATCH_SIZE):
    device = torch.device("cpu")
    input_dim = real_data.shape[1]

    G = Generator(input_dim=input_dim, latent_dim=latent_dim).to(device)
    D = Discriminator(input_dim=input_dim).to(device)

    opt_g = torch.optim.Adam(G.parameters(), lr=1e-4)
    opt_d = torch.optim.Adam(D.parameters(), lr=1e-4)
    criterion = nn.BCELoss()

    # Convert [0,1] -> [-1,1] to match Tanh output
    real_scaled = (real_data * 2.0) - 1.0
    real_tensor = torch.tensor(real_scaled, dtype=torch.float32)

    dataset = TensorDataset(real_tensor)
    loader = DataLoader(dataset, batch_size=min(batch_size, max(8, len(real_tensor))), shuffle=True)

    for _ in range(epochs):
        for (x_real,) in loader:
            bs = x_real.size(0)
            x_real = x_real.to(device)

            # Train D
            z = torch.randn(bs, latent_dim, device=device)
            x_fake = G(z).detach()

            d_real = D(x_real)
            d_fake = D(x_fake)

            loss_d = criterion(d_real, torch.ones_like(d_real)) + criterion(d_fake, torch.zeros_like(d_fake))
            opt_d.zero_grad()
            loss_d.backward()
            opt_d.step()

            # Train G
            z = torch.randn(bs, latent_dim, device=device)
            x_fake = G(z)
            g_fake = D(x_fake)

            loss_g = criterion(g_fake, torch.ones_like(g_fake))
            opt_g.zero_grad()
            loss_g.backward()
            opt_g.step()

    with torch.no_grad():
        z = torch.randn(n_generate, latent_dim, device=device)
        synthetic = G(z).cpu().numpy()

    # Back to [0,1]
    synthetic = (synthetic + 1.0) / 2.0
    synthetic = np.clip(synthetic, 0.0, 1.0)
    return synthetic


def augment_train_data(X_train: pd.DataFrame, y_train: pd.Series, method: str):
    method = method.lower()

    if method == "raw":
        return X_train.copy(), y_train.copy()

    if method in {"smote", "adasyn"}:
        if not HAS_IMBLEARN:
            print(f"[WARN] {method} skipped (imblearn unavailable). Fallback -> raw")
            return X_train.copy(), y_train.copy()

        sampler = SMOTE(random_state=SEED) if method == "smote" else ADASYN(random_state=SEED)
        try:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            return pd.DataFrame(X_res, columns=X_train.columns), pd.Series(y_res)
        except Exception as e:
            print(f"[WARN] {method} failed: {e}. Fallback -> raw")
            return X_train.copy(), y_train.copy()

    if method == "gan":
        if not HAS_TORCH:
            print("[WARN] GAN skipped (torch unavailable). Fallback -> raw")
            return X_train.copy(), y_train.copy()

        X_np = X_train.to_numpy(dtype=np.float32)
        y_np = y_train.to_numpy(dtype=int)

        X_aug_parts, y_aug_parts = [], []
        for cls in [0, 1]:
            cls_data = X_np[y_np == cls]
            if len(cls_data) == 0:
                continue

            n_real = len(cls_data)
            n_target = GAN_TARGET_PER_CLASS
            n_generate = max(0, n_target - n_real)

            if n_generate > 0:
                synth = train_gan_and_generate(cls_data, n_generate=n_generate)
                combined = np.vstack([cls_data, synth])
            else:
                idx = np.random.choice(n_real, n_target, replace=False)
                combined = cls_data[idx]

            X_aug_parts.append(combined)
            y_aug_parts.append(np.full(combined.shape[0], cls, dtype=int))

        X_aug = np.vstack(X_aug_parts)
        y_aug = np.concatenate(y_aug_parts)

        shuffle_idx = np.random.permutation(len(y_aug))
        X_aug = X_aug[shuffle_idx]
        y_aug = y_aug[shuffle_idx]

        return pd.DataFrame(X_aug, columns=X_train.columns), pd.Series(y_aug)

    raise ValueError(f"Unsupported augmentation method: {method}")

In [7]:
def load_shared_gan_fold_artifacts(fold_idx, expected_columns=None):
    x_aug_path = DATA_DIR / f"X_aug_fold_{fold_idx}.csv"
    y_aug_path = DATA_DIR / f"y_aug_fold_{fold_idx}.csv"
    x_test_path = DATA_DIR / f"X_test_scaled_fold_{fold_idx}.csv"

    if not (x_aug_path.exists() and y_aug_path.exists() and x_test_path.exists()):
        return None

    X_aug = pd.read_csv(x_aug_path)
    y_aug = pd.read_csv(y_aug_path).iloc[:, 0].astype(int)
    X_test_scaled = pd.read_csv(x_test_path)

    if expected_columns is not None and list(X_aug.columns) != list(expected_columns):
        print(f"[WARN] Fold {fold_idx} GAN artifact columns differ from current training columns; using exported files anyway.")

    print(f"[GAN] Loaded exported fold artifacts for fold {fold_idx} from {DATA_DIR.resolve()}")
    return X_aug, y_aug, X_test_scaled


def get_fold_data_for_method(X_train, y_train, X_test, method, fold_idx):
    method = method.lower()

    if method == "gan":
        shared = load_shared_gan_fold_artifacts(fold_idx, expected_columns=X_train.columns)
        if shared is not None:
            return shared

        print(f"[WARN] Shared GAN artifacts not found for fold {fold_idx}; falling back to local GAN training.")
        X_aug, y_aug = augment_train_data(X_train, y_train, method="gan")
        return X_aug, y_aug, X_test.copy()

    X_aug, y_aug = augment_train_data(X_train, y_train, method=method)
    return X_aug, y_aug, X_test.copy()

## 7) Baseline Deep Learning: MLP, LSTM, Transformer

- Triển khai các baseline DL (MLP, LSTM, Transformer) theo dạng thích nghi cho dữ liệu tabular.
- Việc dùng cùng một khung evaluate_*_cv giúp đảm bảo quy trình đánh giá nhất quán giữa CML và DL.
- Tất cả metrics đều được tính trên fold test gốc, nhờ đó kết quả phản ánh chất lượng dự báo trên dữ liệu thực tế.

In [8]:
# DL model builders + evaluation (PyTorch)
class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


class LSTMClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.proj = nn.Linear(1, 32)
        encoder_layer = nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=64, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        x = self.proj(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.fc(x)


def fit_torch_model(model, X_train, y_train, X_val, y_val, needs_seq=False, epochs=DL_EPOCHS, batch_size=DL_BATCH_SIZE):
    device = torch.device("cpu")
    model = model.to(device)

    if needs_seq:
        X_train_t = torch.tensor(X_train[..., np.newaxis], dtype=torch.float32)
        X_val_t = torch.tensor(X_val[..., np.newaxis], dtype=torch.float32)
    else:
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        X_val_t = torch.tensor(X_val, dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_val_t = torch.tensor(y_val.reshape(-1, 1), dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=min(batch_size, len(train_ds)), shuffle=True)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_state = None
    best_val = float("inf")
    patience = 4
    no_improve = 0

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t.to(device))
            val_loss = criterion(val_logits, y_val_t.to(device)).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def predict_torch_proba(model, X, needs_seq=False):
    device = torch.device("cpu")
    model.eval()
    with torch.no_grad():
        if needs_seq:
            X_t = torch.tensor(X[..., np.newaxis], dtype=torch.float32).to(device)
        else:
            X_t = torch.tensor(X, dtype=torch.float32).to(device)
        logits = model(X_t)
        prob = torch.sigmoid(logits).cpu().numpy().reshape(-1)
    return prob


def evaluate_cml_cv(X, y, aug_methods=AUG_METHODS):
    records = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for aug in aug_methods:
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_test, y_test = X.iloc[te_idx], y.iloc[te_idx]

            X_aug, y_aug = augment_train_data(X_train, y_train, method=aug)

            for model_name, model in build_cml_models().items():
                clf = clone(model)
                clf.fit(X_aug, y_aug)
                y_prob = get_proba(clf, X_test)
                m = compute_metrics(y_test, y_prob)
                m.update({
                    "dataset": "IBM",
                    "model_type": "CML",
                    "model": model_name,
                    "augmentation": aug,
                    "fold": fold,
                })
                records.append(m)

    return pd.DataFrame(records)


def evaluate_dl_cv(X, y, aug_methods=AUG_METHODS):
    if not HAS_TORCH:
        return pd.DataFrame()

    records = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for aug in aug_methods:
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_test, y_test = X.iloc[te_idx], y.iloc[te_idx]

            X_aug, y_aug = augment_train_data(X_train, y_train, method=aug)

            X_train_arr = X_aug.to_numpy(dtype=np.float32)
            y_train_arr = y_aug.to_numpy(dtype=np.float32)
            X_test_arr = X_test.to_numpy(dtype=np.float32)
            y_test_arr = y_test.to_numpy(dtype=np.float32)

            # Split a validation slice from training fold
            split_idx = int(len(X_train_arr) * 0.85)
            X_tr, y_tr = X_train_arr[:split_idx], y_train_arr[:split_idx]
            X_val, y_val = X_train_arr[split_idx:], y_train_arr[split_idx:]

            dl_builders = {
                "MLP": (MLPClassifier, False),
                "LSTM": (LSTMClassifier, True),
                "Transformer": (TransformerClassifier, True),
            }

            for model_name, (builder, needs_seq) in dl_builders.items():
                model = builder(input_dim=X_train_arr.shape[1])
                model = fit_torch_model(model, X_tr, y_tr, X_val, y_val, needs_seq=needs_seq)
                y_prob = predict_torch_proba(model, X_test_arr, needs_seq=needs_seq)

                m = compute_metrics(y_test_arr, y_prob)
                m.update({
                    "dataset": "IBM",
                    "model_type": "DL",
                    "model": model_name,
                    "augmentation": aug,
                    "fold": fold,
                })
                records.append(m)

    return pd.DataFrame(records)

In [9]:
def evaluate_cml_cv(X, y, aug_methods=AUG_METHODS):
    records = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for aug in aug_methods:
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_test, y_test = X.iloc[te_idx], y.iloc[te_idx]

            X_aug, y_aug, X_test_eval = get_fold_data_for_method(X_train, y_train, X_test, aug, fold)

            for model_name, model in build_cml_models().items():
                clf = clone(model)
                clf.fit(X_aug, y_aug)
                y_prob = get_proba(clf, X_test_eval)
                m = compute_metrics(y_test, y_prob)
                m.update({
                    "dataset": "IBM",
                    "model_type": "CML",
                    "model": model_name,
                    "augmentation": aug,
                    "fold": fold,
                })
                records.append(m)

    return pd.DataFrame(records)


def evaluate_dl_cv(X, y, aug_methods=AUG_METHODS):
    if not HAS_TORCH:
        return pd.DataFrame()

    records = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    for aug in aug_methods:
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
            X_test, y_test = X.iloc[te_idx], y.iloc[te_idx]

            X_aug, y_aug, X_test_eval = get_fold_data_for_method(X_train, y_train, X_test, aug, fold)

            X_train_arr = X_aug.to_numpy(dtype=np.float32)
            y_train_arr = y_aug.to_numpy(dtype=np.float32)
            X_test_arr = X_test_eval.to_numpy(dtype=np.float32)
            y_test_arr = y_test.to_numpy(dtype=np.float32)

            split_idx = int(len(X_train_arr) * 0.85)
            X_tr, y_tr = X_train_arr[:split_idx], y_train_arr[:split_idx]
            X_val, y_val = X_train_arr[split_idx:], y_train_arr[split_idx:]

            dl_builders = {
                "MLP": (MLPClassifier, False),
                "LSTM": (LSTMClassifier, True),
                "Transformer": (TransformerClassifier, True),
            }

            for model_name, (builder, needs_seq) in dl_builders.items():
                model = builder(input_dim=X_train_arr.shape[1])
                model = fit_torch_model(model, X_tr, y_tr, X_val, y_val, needs_seq=needs_seq)
                y_prob = predict_torch_proba(model, X_test_arr, needs_seq=needs_seq)

                m = compute_metrics(y_test_arr, y_prob)
                m.update({
                    "dataset": "IBM",
                    "model_type": "DL",
                    "model": model_name,
                    "augmentation": aug,
                    "fold": fold,
                })
                records.append(m)

    return pd.DataFrame(records)

## 8) Chạy thí nghiệm 5-Fold và tổng hợp metrics

- Đây là cell điều phối thí nghiệm chính cho IBM.
- RUN_FULL_EXPERIMENT=False là chế độ kiểm tra nhanh; khi cần kết quả đầy đủ cho bài báo, bật RUN_FULL_EXPERIMENT=True và RUN_DL_EXPERIMENT=True.
- Bảng summary là kết quả trung bình qua 5 fold, dùng để đối chiếu tác động của từng kỹ thuật augmentation lên từng baseline.

In [10]:
# Run experiments on IBM (focus dataset in the paper)
# Set True when you want to run the full paper-like setting.
RUN_FULL_EXPERIMENT = False
RUN_DL_EXPERIMENT = False

X_ibm, y_ibm = get_xy(ibm_df, target_col="Attrition")

print("IBM feature shape for CV:", X_ibm.shape)
print("IBM class distribution:")
print(y_ibm.value_counts())

if RUN_FULL_EXPERIMENT:
    active_aug_methods = AUG_METHODS
else:
    # Quick sanity mode still includes GAN so the shared-fold ingestion path is exercised.
    active_aug_methods = ["raw", "smote", "gan"]

print("Active augmentation methods:", active_aug_methods)

cml_results = evaluate_cml_cv(X_ibm, y_ibm, aug_methods=active_aug_methods)
print("CML experiments done. Rows:", len(cml_results))

dl_results = pd.DataFrame()
if RUN_DL_EXPERIMENT:
    dl_results = evaluate_dl_cv(X_ibm, y_ibm, aug_methods=active_aug_methods)
    if dl_results.empty:
        print("DL experiments skipped (tensorflow unavailable).")
    else:
        print("DL experiments done. Rows:", len(dl_results))
else:
    print("DL experiments not run in quick mode. Set RUN_DL_EXPERIMENT=True for full DL baselines.")

all_results = pd.concat([cml_results, dl_results], ignore_index=True) if not dl_results.empty else cml_results.copy()

summary = (
    all_results
    .groupby(["dataset", "model_type", "model", "augmentation"], as_index=False)[["accuracy", "precision", "recall", "f1", "auc"]]
    .mean()
    .sort_values(["model_type", "model", "augmentation"])
)

display(summary)

IBM feature shape for CV: (1470, 43)
IBM class distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64
Active augmentation methods: ['raw', 'smote', 'gan']
[WARN] Shared GAN artifacts not found for fold 1; falling back to local GAN training.
[WARN] Shared GAN artifacts not found for fold 2; falling back to local GAN training.
[WARN] Shared GAN artifacts not found for fold 3; falling back to local GAN training.
[WARN] Shared GAN artifacts not found for fold 4; falling back to local GAN training.
[WARN] Shared GAN artifacts not found for fold 5; falling back to local GAN training.
CML experiments done. Rows: 45
DL experiments not run in quick mode. Set RUN_DL_EXPERIMENT=True for full DL baselines.


,dataset,model_type,model,augmentation,accuracy,precision,recall,f1,auc
0,IBM,CML,LogisticRegression,gan,0.854422,0.562090,0.438741,0.490165,0.792834
1,IBM,CML,LogisticRegression,raw,0.883673,0.773443,0.396631,0.523446,0.832634
2,IBM,CML,LogisticRegression,smote,0.768707,0.385705,0.721631,0.502329,0.825892
3,IBM,CML,RandomForest,gan,0.857143,0.822727,0.155940,0.259648,0.803577
4,IBM,CML,RandomForest,raw,0.853741,0.746081,0.155940,0.253818,0.811186
5,IBM,CML,RandomForest,smote,0.865986,0.716333,0.303635,0.422773,0.818651
6,IBM,CML,XGBoost,gan,0.863946,0.685227,0.308156,0.422836,0.819191
7,IBM,CML,XGBoost,raw,0.865306,0.699951,0.299734,0.416161,0.818073
8,IBM,CML,XGBoost,smote,0.864626,0.640948,0.375532,0.469818,0.816757


## 9) Lưu kết quả và kết luận tạm thời

- Cell phía dưới lưu cả kết quả chi tiết theo từng fold và bảng tổng hợp trung bình.
- Hai file trong results/metrics là đầu vào trực tiếp cho phần Experimental Results trong báo cáo hoặc bài báo.
- Khuyến nghị: khi thay đổi augmentation, model hoặc hyperparameters, chạy lại từ cell thí nghiệm chính và lưu đè để đảm bảo nhất quán phiên bản kết quả.

In [11]:
# Save metrics for paper/report
all_results_path = RESULTS_DIR / "ibm_baselines_cv_fold_metrics.csv"
summary_path = RESULTS_DIR / "ibm_baselines_cv_summary_metrics.csv"

all_results.to_csv(all_results_path, index=False)
summary.to_csv(summary_path, index=False)

print("Saved metrics files:")
print("-", all_results_path.resolve())
print("-", summary_path.resolve())

Saved metrics files:
- F:\ML - DL Projects\Deep Learning Employee Attrition BiTCN\results\metrics\ibm_baselines_cv_fold_metrics.csv
- F:\ML - DL Projects\Deep Learning Employee Attrition BiTCN\results\metrics\ibm_baselines_cv_summary_metrics.csv


In [12]:
# Synchronized comparison across notebook 2 (CML) and notebook 3 (Bi-TCN)
bitcn_metrics_path = RESULTS_DIR / "bitcn_cv_fold_metrics.csv"
bitcn_smote_metrics_path = RESULTS_DIR / "bitcn_smote_cv_fold_metrics.csv"
comparison_frames = [summary.assign(source="CML")]


def _load_bitcn_summary(metrics_path: Path, source_label: str):
    if not metrics_path.exists():
        print(f"[INFO] Skipping {source_label}; file not found: {metrics_path.name}")
        return None

    df = pd.read_csv(metrics_path)
    metric_columns = [c for c in ["accuracy", "precision", "recall", "f1", "auc"] if c in df.columns]
    if not metric_columns:
        print(f"[WARN] Skipping {source_label}; no metric columns found in {metrics_path.name}")
        return None

    augmentation = "smote" if "smote" in metrics_path.stem.lower() else "gan"
    summary_row = {
        "dataset": "IBM",
        "model_type": "DL",
        "model": "BiTCN",
        "augmentation": augmentation,
        "source": source_label,
    }
    for metric_name in metric_columns:
        summary_row[metric_name] = pd.to_numeric(df[metric_name], errors="coerce").mean()

    return pd.DataFrame([summary_row])


bitcn_gan_summary = _load_bitcn_summary(bitcn_metrics_path, "BiTCN-GAN")
bitcn_smote_summary = _load_bitcn_summary(bitcn_smote_metrics_path, "BiTCN-SMOTE")

if bitcn_gan_summary is not None:
    comparison_frames.append(bitcn_gan_summary)
if bitcn_smote_summary is not None:
    comparison_frames.append(bitcn_smote_summary)

comparison_summary = pd.concat(comparison_frames, ignore_index=True, sort=False)
comparison_summary = comparison_summary.sort_values(["source", "model_type", "model", "augmentation"], kind="stable")
comparison_output_path = RESULTS_DIR / "synchronized_comparison_summary.csv"
comparison_summary.to_csv(comparison_output_path, index=False)

print("Synchronized comparison summary:")
display(comparison_summary)
print(f"Saved synchronized comparison to {comparison_output_path.resolve()}")

[INFO] Skipping BiTCN-SMOTE; file not found: bitcn_smote_cv_fold_metrics.csv
Synchronized comparison summary:


,dataset,model_type,model,augmentation,accuracy,precision,recall,f1,auc,source
9,IBM,DL,BiTCN,gan,0.825850,0.481436,0.463298,0.462394,0.749727,BiTCN-GAN
0,IBM,CML,LogisticRegression,gan,0.854422,0.562090,0.438741,0.490165,0.792834,CML
1,IBM,CML,LogisticRegression,raw,0.883673,0.773443,0.396631,0.523446,0.832634,CML
2,IBM,CML,LogisticRegression,smote,0.768707,0.385705,0.721631,0.502329,0.825892,CML
3,IBM,CML,RandomForest,gan,0.857143,0.822727,0.155940,0.259648,0.803577,CML
4,IBM,CML,RandomForest,raw,0.853741,0.746081,0.155940,0.253818,0.811186,CML
5,IBM,CML,RandomForest,smote,0.865986,0.716333,0.303635,0.422773,0.818651,CML
6,IBM,CML,XGBoost,gan,0.863946,0.685227,0.308156,0.422836,0.819191,CML
7,IBM,CML,XGBoost,raw,0.865306,0.699951,0.299734,0.416161,0.818073,CML
8,IBM,CML,XGBoost,smote,0.864626,0.640948,0.375532,0.469818,0.816757,CML


Saved synchronized comparison to F:\ML - DL Projects\Deep Learning Employee Attrition BiTCN\results\metrics\synchronized_comparison_summary.csv


## 10. Analysis and Validation Notes

- The baseline notebook now prefers the exported GAN fold artifacts when they are present in `data/processed/`, so Random Forest and XGBoost are evaluated on the same fold-aligned, scaled data used by the Bi-TCN run.
- This makes the GAN comparison fairer than regenerating synthetic samples independently inside each notebook, because the augmentation source and evaluation folds are now synchronized.
- For the paper-level discussion, prioritize Recall and AUC over Accuracy: on this attrition task, Accuracy can look strong even when the model misses most churners.
- In the summary table, compare `raw`, `smote`, and `gan` across CML models first, then contrast those numbers with the Bi-TCN SMOTE and GAN runs to see whether the convolutional sequence bias is helping the neural model recover more true positives without sacrificing AUC.
- If the exported GAN files are missing, the notebook falls back to local GAN training and prints a warning, so the workflow still completes but the strict cross-notebook equivalence is weaker.